# Customer Complaint Classification & Routing Engine
### AMD TCS Hackathon | Agentic AI Solution

---

## Pattern: Single-Agent + FAISS RAG Memory + MCP Tool Registry + Generator-Critic

| Pattern | Role |
|---|---|
| **Single-Agent Orchestration** | One routing agent: perceive -> retrieve -> reason -> act -> persist -> reflect |
| **RAG Memory (FAISS)** | In-memory vector index over 8 SLA policy documents |
| **MCP Tool Registry** | `mcp-server-time` for real-time SLA deadline computation |
| **Generator-Critic Loop** | Critic agent overrides low-confidence routing decisions |

```
Incoming Complaint Stream
        |
        v
+------------------------------------------------------+
|  ROUTING AGENT  (Pydantic-AI + Qwen3 via vLLM)      |
|                                                      |
|  1. retrieve_policies --> FAISS kNN over policies    |
|  2. classify + prioritize + assign team              |
|  3. get_current_time  --> MCP time server            |
|  4. write_case        --> in-memory case store       |
|  5. Critic reflection --> if confidence < 0.70       |
+------------------------------------------------------+
        |
        v
  Live Routing Dashboard (HTML, auto-refreshes)
```

---
## Step 1: Launch vLLM *(run in a separate terminal)*

```bash
VLLM_USE_TRITON_FLASH_ATTN=0 \\
vllm serve Qwen/Qwen3-4B \\
  --served-model-name Qwen3-4B \\
  --api-key abc-123 \\
  --port 8000 \\
  --enable-auto-tool-choice \\
  --tool-call-parser hermes \\
  --trust-remote-code \\
  --max-model-len 8192
```

Monitor GPU: `watch rocm-smi`

> **MI300X tip:** Swap `Qwen3-4B` to `Qwen3-30B-A3B` to utilise the full 192 GB HBM3.

---
## Step 2: Install Dependencies

In [ ]:
!pip install -q pydantic-ai-slim openai faiss-cpu sentence-transformers mcp-server-time pandas

---
## Step 3: Environment Config

In [ ]:
import os

BASE_URL   = "http://localhost:8000/v1"
MODEL_NAME = "Qwen3-4B"  # swap to Qwen3-30B-A3B for full MI300X power

os.environ["BASE_URL"]       = BASE_URL
os.environ["OPENAI_API_KEY"] = "abc-123"

print(f"BASE_URL: {BASE_URL} | MODEL: {MODEL_NAME}")

In [ ]:
# Verify vLLM is live
!curl -s http://localhost:8000/v1/models -H "Authorization: Bearer abc-123" | python3 -m json.tool

---
## Step 4: Build Policy RAG Store (FAISS)

8 SLA + routing policy documents embedded with `all-MiniLM-L6-v2` and indexed in a FAISS `IndexFlatIP`.
Pure in-memory -- no external service, no disk writes, works in the AMD contained environment.

In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

POLICIES = [
    {"id": "pol-001", "text": (
        "BILLING: Payment failures, overcharges, incorrect invoices, refunds, subscription errors. "
        "Route to: Billing Team. "
        "P0 (1h): payment gateway down or mass billing failure >100 accounts. "
        "P1 (4h): individual payment failed, subscription charged incorrectly. "
        "P2 (8h): refund request, invoice dispute. "
        "P3 (24h): billing query, invoice explanation."
    )},
    {"id": "pol-002", "text": (
        "TECHNICAL: System downtime, application errors, bugs, performance issues, API failures, data corruption. "
        "Route to: Technical Support Team. "
        "P0 (30min): production system down, data loss, security breach. "
        "P1 (2h): critical feature broken, API returning 500 errors consistently. "
        "P2 (6h): degraded performance, non-critical feature broken. "
        "P3 (24h): minor UI bug, slow page load."
    )},
    {"id": "pol-003", "text": (
        "ACCOUNT: Login failures, account lockout, password reset, 2FA problems, access permission errors. "
        "Route to: Auth & Identity Team. "
        "P0 (1h): all users locked out, auth service down. "
        "P1 (2h): single user cannot login, account compromised. "
        "P2 (6h): password reset not working, 2FA issues. "
        "P3 (24h): account settings query, profile update."
    )},
    {"id": "pol-004", "text": (
        "FRAUD: Unauthorized transactions, account takeover, phishing, identity theft, chargeback fraud. "
        "Route to: Fraud & Risk Team. "
        "P0 (15min): active unauthorized transaction or large-scale fraud in progress. "
        "P1 (1h): unauthorized charge, suspected account takeover. "
        "P2 (4h): suspicious login, phishing report. "
        "P3 (12h): past suspicious activity review."
    )},
    {"id": "pol-005", "text": (
        "SHIPPING: Delayed delivery, lost package, damaged goods, wrong item, tracking not updating. "
        "Route to: Logistics & Fulfillment Team. "
        "P0 (2h): high-value item lost, time-sensitive delivery missed. "
        "P1 (4h): package not delivered, tracking shows delivered but not received. "
        "P2 (8h): delayed package, wrong item received. "
        "P3 (48h): minor packaging complaint, general delivery inquiry."
    )},
    {"id": "pol-006", "text": (
        "FEEDBACK: Feature requests, suggestions, compliments, general improvement ideas. "
        "Route to: Product Team. "
        "P3 (72h): all feedback is P3 by default. "
        "Escalation: legal threat or media mention -> bump to P1, CC Legal/PR Team."
    )},
    {"id": "pol-007", "text": (
        "ESCALATION RULES: "
        "1. VIP/Enterprise customer -> bump priority one level. "
        "2. legal, lawsuit, GDPR, data breach -> CC Legal Team. "
        "3. social media, Twitter, viral, press -> CC PR Team. "
        "4. Same issue repeated >2 times in 7 days -> minimum P1. "
        "5. Case at 80% of SLA window unresolved -> auto-escalate to P0."
    )},
    {"id": "pol-008", "text": (
        "TEAM CONTACTS: "
        "Billing Team: billing@support.internal | Slack: #billing-ops. "
        "Technical Support Team: tech@support.internal | Slack: #tech-support-l1. "
        "Auth & Identity Team: auth@support.internal | Slack: #identity-ops. "
        "Fraud & Risk Team: fraud@support.internal | Slack: #fraud-alerts | PagerDuty: P0 auto-page. "
        "Logistics & Fulfillment Team: logistics@support.internal | Slack: #fulfillment. "
        "Product Team: product@support.internal | Slack: #product-feedback. "
        "Legal Team (CC only): legal@internal.com. "
        "PR Team (CC only): pr@internal.com."
    )},
]

POLICY_TEXTS = [p["text"] for p in POLICIES]

# Embed all 8 docs with MiniLM (CPU, fast for small corpora)
EMBED_MODEL = SentenceTransformer("all-MiniLM-L6-v2")
policy_vecs = EMBED_MODEL.encode(POLICY_TEXTS, normalize_embeddings=True).astype("float32")

# IndexFlatIP = exact cosine similarity (L2-normalised vectors)
FAISS_INDEX = faiss.IndexFlatIP(policy_vecs.shape[1])
FAISS_INDEX.add(policy_vecs)

print(f"FAISS index ready | {FAISS_INDEX.ntotal} docs | dim={policy_vecs.shape[1]}")

---
## Step 5: Define Agent Tools & Case Store

In [ ]:
import pandas as pd
import uuid
from datetime import datetime, timedelta
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic_ai.mcp import MCPServerStdio
from IPython.display import display, clear_output, HTML
import asyncio

CASE_STORE: list[dict] = []
PRIORITY_COLORS = {"P0": "#dc2626", "P1": "#ea580c", "P2": "#ca8a04", "P3": "#16a34a"}
PRIORITY_SLA    = {"P0": 0.5, "P1": 2.0, "P2": 6.0, "P3": 24.0}


def retrieve_policies(query: str) -> str:
    """Retrieve top-3 SLA/routing policies for the given complaint keywords.

    Args:
        query: 3-7 keywords from the complaint (e.g. payment failed billing).

    Returns:
        Top-3 matching policy texts joined by separators.
    """
    q_vec = EMBED_MODEL.encode([query], normalize_embeddings=True).astype("float32")
    _, indices = FAISS_INDEX.search(q_vec, k=3)
    docs = [POLICY_TEXTS[i] for i in indices[0] if i >= 0]
    return "\n\n---\n".join(docs) if docs else "No policy found. Default: P3 / General Support."


def write_case(
    complaint_text: str,
    category: str,
    priority: str,
    team: str,
    sla_hours: float,
    cc_teams: str,
    confidence_score: float,
    explanation: str,
) -> str:
    """Persist a classified and routed complaint to the live dashboard store.

    Args:
        complaint_text: Raw complaint (truncated to 120 chars).
        category: billing | technical | account | fraud | shipping | feedback | other.
        priority: P0 | P1 | P2 | P3.
        team: Primary team assigned.
        sla_hours: Hours from now to resolve.
        cc_teams: Comma-separated CC teams, or empty string.
        confidence_score: Routing confidence 0.0 to 1.0.
        explanation: One-sentence routing rationale.

    Returns:
        case_id: Unique case identifier.
    """
    now     = datetime.now()
    case_id = f"CASE-{str(uuid.uuid4())[:8].upper()}"
    sla_due = now + timedelta(hours=sla_hours)
    CASE_STORE.append({
        "Case ID":     case_id,
        "Created At":  now.strftime("%Y-%m-%d %H:%M"),
        "Complaint":   complaint_text[:120] + ("..." if len(complaint_text) > 120 else ""),
        "Category":    category.upper(),
        "Priority":    priority,
        "Team":        team,
        "CC Teams":    cc_teams or "no CC",
        "SLA Hours":   sla_hours,
        "SLA Due By":  sla_due.strftime("%Y-%m-%d %H:%M"),
        "Confidence":  f"{confidence_score:.0%}",
        "Explanation": explanation,
        "Status":      "OPEN",
    })
    return case_id

print("Tools ready: retrieve_policies (FAISS), write_case")

---
## Step 6: Build the Routing Agent

In [ ]:
provider  = OpenAIProvider(base_url=os.environ["BASE_URL"], api_key=os.environ["OPENAI_API_KEY"])
llm_model = OpenAIModel(MODEL_NAME, provider=provider)

# MCP time server -- real-time clock for SLA deadline computation
time_server = MCPServerStdio(
    "python",
    args=["-m", "mcp_server_time", "--local-timezone", "Asia/Kolkata"],
)

ROUTING_SYSTEM_PROMPT = """
You are an expert customer complaint triage and routing agent.

For every complaint, strictly follow this 6-step sequence:
1. Call retrieve_policies with a 3-7 keyword query derived from the complaint.
2. Classify into ONE category: billing | technical | account | fraud | shipping | feedback | other.
3. Assign priority P0/P1/P2/P3 using the retrieved SLA rules.
4. Determine primary team and any CC teams (Legal, PR per escalation rules).
5. Call get_current_time, then compute sla_hours from the policy.
6. Call write_case with all fields. Report confidence_score honestly (0.0-1.0).

Escalation triggers:
- legal, lawsuit, GDPR, data breach -> CC Legal Team
- social media, Twitter, viral, press -> CC PR Team
- VIP/Enterprise keywords -> bump priority one level
- Fraud/security -> minimum P1 always

After tool calls, provide a brief 2-sentence routing summary.
"""

routing_agent = Agent(
    model=llm_model,
    system_prompt=ROUTING_SYSTEM_PROMPT,
    tools=[retrieve_policies, write_case],
    mcp_servers=[time_server],
)

print(f"Routing Agent ready | Model: {MODEL_NAME}")
print("Tools: retrieve_policies (FAISS), write_case, get_current_time (MCP)")

---
## Step 7: Live Routing Dashboard Renderer

In [ ]:
def render_dashboard():
    """Render CASE_STORE as a styled HTML live routing dashboard."""
    if not CASE_STORE:
        print("No cases routed yet.")
        return

    df = pd.DataFrame(CASE_STORE)
    df["_rank"] = df["Priority"].map({"P0":0,"P1":1,"P2":2,"P3":3})
    df = df.sort_values(["_rank","SLA Due By"]).drop(columns=["_rank"])

    rows_html = ""
    for _, row in df.iterrows():
        color = PRIORITY_COLORS.get(row["Priority"], "#6b7280")
        rows_html += (
            "<tr>"
            f'<td style="font-family:monospace;font-size:12px">{row["Case ID"]}</td>'
            f'<td style="font-size:12px;color:#6b7280">{row["Created At"]}</td>'
            f'<td style="max-width:200px;font-size:12px">{row["Complaint"]}</td>'
            f'<td><span style="background:#f1f5f9;border-radius:4px;padding:2px 8px;font-size:12px;font-weight:600">{row["Category"]}</span></td>'
            f'<td><span style="background:{color};color:white;border-radius:4px;padding:2px 8px;font-size:12px;font-weight:700">{row["Priority"]}</span></td>'
            f'<td style="font-size:12px;font-weight:600">{row["Team"]}</td>'
            f'<td style="font-size:11px;color:#6b7280">{row["CC Teams"]}</td>'
            f'<td style="font-size:12px;color:{color};font-weight:600">{row["SLA Hours"]}h</td>'
            f'<td style="font-size:12px;font-family:monospace">{row["SLA Due By"]}</td>'
            f'<td style="font-size:12px">{row["Confidence"]}</td>'
            f'<td style="font-size:11px;color:#6b7280;max-width:200px">{row["Explanation"]}</td>'
            f'<td><span style="background:#dbeafe;color:#1d4ed8;border-radius:4px;padding:2px 8px;font-size:11px">{row["Status"]}</span></td>'
            "</tr>"
        )

    summary = df["Priority"].value_counts().to_dict()
    summary_html = "".join([
        f'<span style="background:{PRIORITY_COLORS.get(p,"#6b7280")};color:white;border-radius:6px;'
        f'padding:4px 12px;font-size:14px;font-weight:700;margin-right:8px">{p}: {summary.get(p,0)}</span>'
        for p in ["P0","P1","P2","P3"]
    ])

    headers = ["Case ID","Created","Complaint","Category","Priority","Team","CC","SLA","Due By","Confidence","Explanation","Status"]
    th_html  = "".join(f'<th style="padding:10px 12px;font-size:12px">{h}</th>' for h in headers)

    html = (
        '<div style="font-family:Segoe UI,sans-serif">'
        '<div style="display:flex;align-items:center;justify-content:space-between;margin-bottom:16px">'
        '<div><h2 style="margin:0;color:#0f172a;font-size:20px">Live Complaint Routing Dashboard</h2>'
        f'<p style="margin:4px 0 0;color:#64748b;font-size:13px">AMD TCS Hackathon | {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}</p></div>'
        f'<div style="text-align:right"><div style="font-size:24px;font-weight:700">{len(df)}</div>'
        '<div style="font-size:12px;color:#64748b">Total Cases</div></div></div>'
        f'<div style="margin-bottom:16px">{summary_html}</div>'
        '<div style="overflow-x:auto">'
        '<table style="width:100%;border-collapse:collapse;background:white;border-radius:8px;overflow:hidden;box-shadow:0 1px 3px rgba(0,0,0,.1)">'
        f'<thead><tr style="background:#1e293b;color:white;text-align:left">{th_html}</tr></thead>'
        f'<tbody>{rows_html}</tbody>'
        '</table></div></div>'
    )
    clear_output(wait=True)
    display(HTML(html))

print("Dashboard renderer ready")

---
## Step 8: Async Run Helper

In [ ]:
async def route_complaint(complaint_text: str) -> str:
    """Route a single complaint through the full agent pipeline."""
    prompt = (
        "Route this customer complaint:\n\n"
        f"COMPLAINT:\n{complaint_text}\n\n"
        "Follow the 6-step process: retrieve_policies -> classify -> prioritize -> compute SLA -> write_case -> summarize."
    )
    async with routing_agent.run_mcp_servers():
        result = await routing_agent.run(prompt)
    return result.output

print("route_complaint helper ready")

---
## Step 9: Generator-Critic Reflection Loop

If `confidence_score < 0.70`, a dedicated critic agent reviews the generator's decision
and can override `priority`, `team`, and `sla_hours` in-place.
One pass -- aligns with the slide's *'most gains in first 2-3 iterations'* principle.

In [ ]:
CRITIC_PROMPT = """
You are a senior routing quality critic for a customer support triage system.

You will receive the original complaint and the generator routing decision.
- If correct: reply exactly -> APPROVED: <brief reason>
- If wrong:   reply exactly -> CORRECTED: <new_priority>|<new_team>|<new_sla_hours>|<reason>
  Example: CORRECTED: P0|Fraud & Risk Team|1.0|Active fraud requires immediate escalation

Rules: fraud/security never below P1; production outages always P0; billing disputes rarely above P2.
"""

critic_agent = Agent(model=llm_model, system_prompt=CRITIC_PROMPT)


async def route_with_reflection(complaint_text: str, confidence_threshold: float = 0.70) -> str:
    """Generator pass followed by optional Critic pass if confidence < threshold."""
    prompt = (
        "Route this customer complaint:\n\n"
        f"COMPLAINT:\n{complaint_text}\n\n"
        "Follow the 6-step process: retrieve_policies -> classify -> prioritize -> compute SLA -> write_case -> summarize."
    )

    # Generator
    async with routing_agent.run_mcp_servers():
        gen_result = await routing_agent.run(prompt)

    if not CASE_STORE:
        return gen_result.output

    last = CASE_STORE[-1]
    confidence = float(last["Confidence"].strip("%")) / 100.0
    print(f"   Generator confidence: {last['Confidence']}")

    # Critic -- only triggered on low confidence
    if confidence < confidence_threshold:
        print("   Low confidence -> Critic reviewing...")
        critic_input = (
            f"Original Complaint: {complaint_text}\n\n"
            "Generator Decision:\n"
            f"- Category: {last['Category']}\n"
            f"- Priority: {last['Priority']}\n"
            f"- Team: {last['Team']}\n"
            f"- SLA: {last['SLA Hours']}h\n"
            f"- Explanation: {last['Explanation']}\n\n"
            "Approve or correct this routing decision."
        )
        critic_out = (await critic_agent.run(critic_input)).output.strip()

        if critic_out.startswith("CORRECTED:"):
            parts = critic_out.replace("CORRECTED:", "").strip().split("|")
            if len(parts) >= 4:
                try:
                    new_p = parts[0].strip()
                    new_t = parts[1].strip()
                    new_sla = float(parts[2].strip())
                    reason = parts[3].strip()
                    old_p = last["Priority"]
                    CASE_STORE[-1].update({
                        "Priority": new_p, "Team": new_t, "SLA Hours": new_sla,
                        "Explanation": f"[Critic: {old_p}->{new_p}] {reason}"
                    })
                    print(f"   Corrected: {old_p} -> {new_p} | {new_t}")
                except ValueError:
                    print("   Critic parse error -- keeping generator decision.")
        else:
            print("   Critic approved.")

    return gen_result.output

print("Generator-Critic reflection loop ready")

---
## Step 10: Incoming Complaint Stream (Sample Data)

In [ ]:
INCOMING_COMPLAINTS = [
    # P0 -- Technical (production down, enterprise)
    "Our entire production system is down! All users are getting 503 errors for the past 30 minutes. "
    "We are losing approximately $50,000 per hour. This is an enterprise account. URGENT!",

    # P0 -- Fraud (active unauthorized transaction)
    "I just saw 3 unauthorized transactions totalling $4,500. "
    "Someone is transferring money from my account right now. Please freeze it immediately!",

    # P1 -- Technical (API down, checkout broken)
    "The payment API has been returning 500 errors for 2 hours. "
    "Our checkout is completely broken and we cannot process any customer orders.",

    # P1 -- Account (cannot login before important event)
    "I cannot log in since yesterday. Password reset emails are not arriving. "
    "I have a critical client presentation tomorrow that requires dashboard access.",

    # P1 -- Billing + legal threat -> CC Legal
    "You charged my credit card 3 times for the same invoice! Overcharge is $1,200. "
    "If not refunded in 24 hours, I will take legal action and report this GDPR violation.",

    # P2 -- Shipping (medical equipment delayed)
    "My order #ORD-78432 was due 3 days ago. Tracking shows it left the warehouse but has not updated. "
    "The package contains medical equipment urgently needed by my patient.",

    # P2 -- Technical (mobile app crash since update)
    "The mobile app crashes every time I export reports since update v3.2.1. "
    "Desktop works fine. iPhone 14, iOS 17.4.",

    # P2 -- Billing (refund for cancelled subscription)
    "I cancelled my subscription 3 weeks ago but was still charged $99 this month. "
    "Order ID: SUB-44291. Please refund to my original payment method.",

    # P1 -- Fraud + social media threat -> CC PR
    "I received a phishing email appearing to come from your company. Colleagues clicked the link. "
    "If you do not respond immediately, I am posting this on Twitter and contacting the press.",

    # P3 -- Feedback
    "Overall great product but dark mode has contrast issues in the settings panel. "
    "CSV export for the reports section would be very helpful.",

    # P3 -- Billing query
    "Can you send me an itemized invoice for May 2026? Needed for accounting. Account: john.doe@company.com",

    # P2 -- Account (2FA delay)
    "Two-factor authentication is not working. SMS codes arrive 20-30 minutes late and expire before use."
]

print(f"{len(INCOMING_COMPLAINTS)} complaints loaded for routing stream")

---
## Step 11: Run the Full Routing Stream

In [ ]:
CASE_STORE.clear()
print('Case store cleared.')

In [ ]:
async def run_stream():
    total = len(INCOMING_COMPLAINTS)
    for i, complaint in enumerate(INCOMING_COMPLAINTS, 1):
        print(f"\n{\"="*60}")
        print(f"Case {i}/{total}: {complaint[:80]}{\"...\" if len(complaint)>80 else \"\"}")
        try:
            await route_with_reflection(complaint, confidence_threshold=0.70)
        except Exception as e:
            print(f"Error: {e}")
            continue
        render_dashboard()
        if i < total:
            await asyncio.sleep(1.0)  # simulate stream delay
    print(f"\nStream complete -- {len(CASE_STORE)} cases routed.")
    render_dashboard()

await run_stream()

---
## Step 12: Route a Custom Complaint (Interactive)

In [ ]:
my_complaint = (
    "I have been trying to get a refund for 3 weeks. "
    "Every time I call, I am transferred between teams with no resolution. "
    "The amount is $350 for a product returned in perfect condition. "
    "I am considering disputing this with my bank."
)

result = await route_with_reflection(my_complaint)
print(f"\nAgent Summary:\n{result}")
render_dashboard()

---
## Step 13: Analytics

In [ ]:
if CASE_STORE:
    df = pd.DataFrame(CASE_STORE)
    print("ROUTING ANALYTICS\n" + "="*40)
    print("\nBy Priority:\n", df["Priority"].value_counts().to_string())
    print("\nBy Category:\n", df["Category"].value_counts().to_string())
    print("\nBy Team:\n",     df["Team"].value_counts().to_string())
    print("\nAvg SLA by Priority:\n", df.groupby("Priority")["SLA Hours"].mean().sort_index().to_string())
    p0 = len(df[df["Priority"]=="P0"])
    if p0: print(f"\n{p0} CRITICAL (P0) case(s) need immediate attention!")
else:
    print("No cases -- run Step 11 first.")

---
## Step 14: Export to CSV

In [ ]:
if CASE_STORE:
    ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = f"./routed_cases_{ts}.csv"
    pd.DataFrame(CASE_STORE).to_csv(path, index=False)
    print(f"Exported {len(CASE_STORE)} cases -> {path}")
    print("Right-click the file in JupyterLab -> Download")
else:
    print("No cases to export.")

---
## Solution Summary

| Component | Implementation |
|---|---|
| **LLM** | Qwen3-4B / Qwen3-30B-A3B via vLLM on AMD MI300X |
| **Framework** | Pydantic-AI (OpenAI-compatible endpoint) |
| **Agentic Pattern** | Single-Agent Orchestration + Generator-Critic Reflection |
| **Memory** | FAISS `IndexFlatIP` + `all-MiniLM-L6-v2` (fully in-memory, no external service) |
| **Tools** | `retrieve_policies` (FAISS), `write_case`, `get_current_time` (MCP) |
| **Dashboard** | Live HTML table sorted by Priority x SLA deadline |
| **Hardware** | AMD MI300X -- 192 GB HBM3, 5.3 TB/s bandwidth |

## Future Work
- **Mem0** long-term memory: auto-escalate repeat complainants across sessions
- **Parallel Fan-Out**: specialist sub-agents per category
- **Human-in-the-Loop**: P0 auto-trigger PagerDuty + Slack approval workflow
- **Fine-tuning**: distil routing decisions into a small SLM for ultra-low latency